**Получение данных**

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine

In [ ]:
# подгружаем .env
load_dotenv()

In [ ]:
# Считываем все креды
src_host = os.environ.get('DB_SOURCE_HOST')
src_port = os.environ.get('DB_SOURCE_PORT')
src_username = os.environ.get('DB_SOURCE_USER')
src_password = os.environ.get('DB_SOURCE_PASSWORD')
src_db = os.environ.get('DB_SOURCE_NAME')

dst_host = os.environ.get('DB_DESTINATION_HOST')
dst_port = os.environ.get('DB_DESTINATION_PORT')
dst_username = os.environ.get('DB_DESTINATION_USER')
dst_password = os.environ.get('DB_DESTINATION_PASSWORD')
dst_db = os.environ.get('DB_DESTINATION_NAME')

s3_bucket = os.environ.get('S3_BUCKET_NAME')
s3_access_key = os.environ.get('AWS_ACCESS_KEY_ID')
s3_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')

In [ ]:
# Создадим соединения
# src_conn = create_engine(f'postgresql://{src_username}:{src_password}@{src_host}:{src_port}/{src_db}')
dst_conn = create_engine(f'postgresql://{dst_username}:{dst_password}@{dst_host}:{dst_port}/{dst_db}')

In [17]:
data = pd.read_sql("""
select * from real_estate_clean
""", dst_conn)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123661 entries, 0 to 123660
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 123661 non-null  int64  
 1   flat_id            123661 non-null  int64  
 2   build_year         123661 non-null  int64  
 3   building_type_int  123661 non-null  int64  
 4   latitude           123661 non-null  float64
 5   longitude          123661 non-null  float64
 6   ceiling_height     123661 non-null  float64
 7   flats_count        123661 non-null  int64  
 8   floors_total       123661 non-null  int64  
 9   has_elevator       123661 non-null  int64  
 10  floor              123661 non-null  int64  
 11  kitchen_area       123661 non-null  float64
 12  living_area        123661 non-null  float64
 13  rooms              123661 non-null  int64  
 14  is_apartment       123661 non-null  int64  
 15  studio             123661 non-null  int64  
 16  to

In [18]:
data.head()

,id,flat_id,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price
0,1,0,1965,6,55.717113,37.781120,2.64,84,12,1,9,9.9,19.900000,1,0,0,35.099998,9500000
1,2,1,2001,2,55.794849,37.608013,3.00,97,10,1,7,0.0,16.600000,1,0,0,43.000000,13500000
2,3,2,2000,4,55.740040,37.761742,2.70,80,10,1,9,9.0,32.000000,2,0,0,56.000000,13500000
3,4,3,2002,4,55.672016,37.570877,2.64,771,17,1,1,10.1,43.099998,3,0,0,76.000000,20000000
4,5,4,1971,1,55.808807,37.707306,2.60,208,9,1,3,3.0,14.000000,1,0,0,24.000000,5200000


**Обучение базовой модели**

In [19]:
target = 'price'
num_features = ['build_year', 'latitude', 'longitude', 'ceiling_height', 'flats_count', 'floors_total', 'floor', 'kitchen_area', 'living_area', 'rooms', 'total_area']
cat_features = ['building_type_int']
cat_bin_features = ['has_elevator', 'is_apartment', 'studio']

In [21]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    [
        ('binary', OneHotEncoder(drop='if_binary', sparse_output=False), cat_bin_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features),
        ('num', StandardScaler(), num_features)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

In [22]:
pd.DataFrame(preprocessor.fit_transform(data), columns=preprocessor.get_feature_names_out()).head()

,has_elevator_1,is_apartment_1,studio_0,building_type_int_1,building_type_int_2,building_type_int_3,building_type_int_4,building_type_int_5,building_type_int_6,build_year,latitude,longitude,ceiling_height,flats_count,floors_total,floor,kitchen_area,living_area,rooms,total_area
0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.972862,-0.130952,1.271414,-0.534770,-0.809754,-0.298637,0.273462,0.190837,-0.484529,-1.145860,-0.718647
1,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.664237,0.622453,0.123194,1.228443,-0.746818,-0.590135,-0.078080,-1.753635,-0.630780,-1.145860,-0.502225
2,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.618762,0.091247,1.142875,-0.240901,-0.829118,-0.590135,0.273462,0.014067,0.051724,-0.118838,-0.146089
3,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.709712,-0.568028,-0.123130,-0.534770,2.516139,0.430108,-1.132708,0.230119,0.543658,0.908184,0.401814
4,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,-0.700012,0.757732,0.781803,-0.730683,-0.209447,-0.735884,-0.781165,-1.164401,-0.746008,-1.145860,-1.022732


In [24]:
from sklearn.model_selection import train_test_split
RANDOM_STATE = 101

X_tr, X_val, y_tr, y_val = train_test_split(data.drop(target, axis=1), data[target], random_state=RANDOM_STATE)
X_tr_prepared = preprocessor.fit_transform(X_tr, y_tr)
X_val_prepared = preprocessor.transform(X_val)

In [25]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import numpy as np
import warnings
warnings.filterwarnings('ignore')

models = {
    "Linear Regressor": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    "LightGBM": LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    "XGBoost": XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    "CatBoost": CatBoostRegressor(random_state=RANDOM_STATE, verbose=0)
}

print(f"{'Модель':<16} | {'R2':<6} | {'MAE (цена)':<12} | {'RMSE (цена)':<12} | {'MAPE (%)':<8}")
print("-" * 75)
for name, model in models.items():
    model.fit(X_tr_prepared, y_tr)
    y_pred = model.predict(X_val_prepared)
    # Расчет метрик
    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mape = mean_absolute_percentage_error(y_val, y_pred) * 100 # Умножаем на 100 для красивых процентов
    # Вывод результатов
    print(f"{name:<16} | {r2:.4f} | {mae:<12.0f} | {rmse:<12.0f} | {mape:<8.2f}")

Модель           | R2     | MAE (цена)   | RMSE (цена)  | MAPE (%)
---------------------------------------------------------------------------
Linear Regressor | 0.6292 | 7727503      | 20171178     | 44.65   
Random Forest    | 0.8224 | 3471122      | 13961435     | 17.30   
LightGBM         | 0.8235 | 3910353      | 13914627     | 19.81   
XGBoost          | 0.8269 | 3835200      | 13782605     | 19.29   
CatBoost         | 0.8331 | 3796685      | 13534661     | 19.34   


В качестве базовой модели для дальнейшей настройки выбран CatBoost. Алгоритм продемонстрировал наивысшую общую точность (R2: 0.83), однако определяющим фактором стало минимальное значение RMSE. Это подтверждает, что CatBoost обладает высокой устойчивостью к ценовым выбросам и надежно минимизирует крупные ошибки при оценке элитных или нестандартных объектов.

Объединим обучение и трансформацию в единый пайплайн

In [27]:
from sklearn.pipeline import Pipeline

model = CatBoostRegressor(random_state=RANDOM_STATE, verbose=0)
pipeline = Pipeline(
	[
        ('transform', preprocessor),
        ('model', model)
    ]
)
pipeline.fit(X_tr, y_tr)
y_pred = pipeline.predict(X_val)
print(f"R2 пайплайна на валидации: {r2_score(y_val, y_pred):.4f}")

R2 пайплайна на валидации: 0.8331


Проведём кросс-валидацию

In [29]:
from sklearn.model_selection import KFold, cross_validate

cv_strategy = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_res = cross_validate(
    pipeline,
    X_tr,
    y_tr,
    scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'],
    cv=cv_strategy,
    n_jobs=-1,
    error_score='raise'
)

for key, value in cv_res.items():
    print(f"{key:<50}: {sum(value)/len(value)}")


fit_time                                          : 30.682176446914674
score_time                                        : 0.09255189895629883
test_r2                                           : 0.8681559546024061
test_neg_mean_absolute_error                      : -3662670.1244409294
test_neg_root_mean_squared_error                  : -11534600.03026342


Сохранение модели

In [30]:
import joblib
pipeline.fit(X_tr, y_tr)

with open('fitted_model.pkl', 'wb') as fd:
    joblib.dump(pipeline, fd)